In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import BartForConditionalGeneration, BartTokenizer, Trainer, TrainingArguments

W0321 11:26:12.252000 6372 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# ----------------------------
# Step 1: Load Data
# ----------------------------
train_df = pd.read_csv("dataset/public/train.csv")
test_df = pd.read_csv("dataset/public/test.csv")

In [3]:
# ----------------------------
# Step 2: Prepare Dataset Class
# ----------------------------
class DeltaDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.data = df
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Input: Original Prompt + Target Output
        input_text = f"source: {self.data.iloc[idx]['original_prompt']} target: {self.data.iloc[idx]['target_output']}"
        # Label: The Instruction we want to learn
        label_text = self.data.iloc[idx]['instructional_delta']

        model_inputs = self.tokenizer(input_text, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        labels = self.tokenizer(label_text, max_length=128, padding="max_length", truncation=True, return_tensors="pt")

        return {
            "input_ids": model_inputs["input_ids"].squeeze(),
            "attention_mask": model_inputs["attention_mask"].squeeze(),
            "labels": labels["input_ids"].squeeze()
        }

# Initialize Model and Tokenizer
model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

train_dataset = DeltaDataset(train_df, tokenizer)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [4]:
# ----------------------------
# Step 3: Fine-Tuning (The "Score Booster")
# ----------------------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,           # Adjust based on local compute time
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.01,
    # fp16=torch.cuda.is_available(), # Use mixed precision if GPU is available
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print("Starting Fine-Tuning...")
trainer.train()

Starting Fine-Tuning...


C:\Users\ttarv\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
100,5.033333
200,0.960316
300,0.465629
400,0.451331
500,0.406621
600,0.430724
700,0.405879
800,0.391856
900,0.395513
1000,0.399069


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2953, training_loss=0.5554152171247658, metrics={'train_runtime': 89371.5356, 'train_samples_per_second': 0.132, 'train_steps_per_second': 0.033, 'total_flos': 3600493785907200.0, 'train_loss': 0.5554152171247658, 'epoch': 1.0})

In [5]:
# ----------------------------
# Step 4: Generate Predictions
# ----------------------------
def generate_predictions(test_df, model, tokenizer):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    
    preds = []
    for _, row in test_df.iterrows():
        input_text = f"source: {row['original_prompt']} target: {row['target_output']}"
        inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        
        # Use Beam Search for higher quality
        outputs = model.generate(inputs["input_ids"], num_beams=5, max_length=128, early_stopping=True)
        preds.append(tokenizer.decode(outputs[0], skip_special_tokens=True))
    
    return preds

print("Generating test predictions...")
predictions = generate_predictions(test_df, model, tokenizer)
print("Prediction generation complete!")

Generating test predictions...
Prediction generation complete!


In [6]:
# ----------------------------
# Step 5: Create Submission
# ----------------------------
submission = pd.DataFrame({
    "row_id": test_df["row_id"],
    "instructional_delta": predictions
})
submission.to_csv("working/submission.csv", index=False)
print("Submission saved to submission.csv")

Submission saved to submission.csv
